In [0]:
%pip install xgboost
dbutils.library.restartPython()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

import pandas as pd
import numpy as np

from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import mlflow
import mlflow.xgboost

In [0]:
sales = spark.table("cscie103_catalog.final_project.sales")

test_raw = (
    spark.read.csv(
        "/Volumes/cscie103_catalog/final_project/data/test.csv",
        header=True,
        inferSchema=True
    )
    .withColumn("date", F.to_date("date"))
    .withColumnRenamed("family", "product_family")
)

test = (
    test_raw
    .withColumn("onpromotion", F.col("onpromotion").cast("int"))
    .select("id", "date", "store_nbr", "product_family", "onpromotion")
)

In [0]:
min_date = sales.select(F.min("date")).first()[0]
max_test_date = test.select(F.max("date")).first()[0]

date_df = (
    spark.range(0, (max_test_date - min_date).days + 1)
    .withColumn("date", F.expr(f"date_add('{min_date}', CAST(id AS INT))"))
    .select("date")
)

store_fam = sales.select("store_nbr", "product_family").distinct()

calendar = store_fam.crossJoin(date_df)


In [0]:
calendar = (
    calendar
    .join(
        sales.select(
            "date",
            "store_nbr",
            "product_family",
            "sales",
            "onpromotion",
            "store_type",
            "store_cluster",
            "city",
            "state",
            "daily_transactions"
        ),
        ["date", "store_nbr", "product_family"],
        "left"
    )
)

In [0]:
calendar = (
    calendar
    .join(
        spark.table("cscie103_catalog.final_project.oil_prices")
             .select(F.col("date"), F.col("oil_price").alias("oil_price_clean")),
        "date",
        "left"
    )
    .join(
        spark.table("cscie103_catalog.final_project.holidays").select(
            F.col("date"),
            F.col("is_national_holiday").alias("is_national_holiday_clean"),
            F.col("is_holiday").alias("is_holiday_clean"),
            F.col("is_event").alias("is_event_clean"),
            F.col("is_special_day").alias("is_special_day_clean")
        ),
        "date",
        "left"
    )
)

In [0]:
w = Window.partitionBy("store_nbr", "product_family").orderBy("date")

calendar_ff = (
    calendar
    .withColumn("sales_ff", F.last("sales", ignorenulls=True).over(w))
    .withColumn("onpromotion_ff", F.last("onpromotion", ignorenulls=True).over(w))
    .withColumn("oil_price_ff", F.last("oil_price_clean", ignorenulls=True).over(w))
    .withColumn("is_national_holiday_ff", F.last("is_national_holiday_clean", ignorenulls=True).over(w))
    .withColumn("is_holiday_ff", F.last("is_holiday_clean", ignorenulls=True).over(w))
    .withColumn("is_event_ff", F.last("is_event_clean", ignorenulls=True).over(w))
    .withColumn("is_special_day_ff", F.last("is_special_day_clean", ignorenulls=True).over(w))
    .withColumn("daily_transactions_ff", F.last("daily_transactions", ignorenulls=True).over(w))
)

lags = (
    calendar_ff
    .withColumn("sales_lag_1", F.lag("sales_ff", 1).over(w))
    .withColumn("sales_lag_7", F.lag("sales_ff", 7).over(w))
    .withColumn("sales_lag_14", F.lag("sales_ff", 14).over(w))
    .withColumn("sales_lag_28", F.lag("sales_ff", 28).over(w))
    .withColumn("sales_lag_30", F.lag("sales_ff", 30).over(w))
    .withColumn("sales_lag_60", F.lag("sales_ff", 60).over(w))
    .withColumn("sales_lag_90", F.lag("sales_ff", 90).over(w))
)

roll = (
    lags
    .withColumn("rolling_mean_7", F.avg("sales_ff").over(w.rowsBetween(-7, -1)))
    .withColumn("rolling_mean_28", F.avg("sales_ff").over(w.rowsBetween(-28, -1)))
    .withColumn("rolling_mean_90", F.avg("sales_ff").over(w.rowsBetween(-90, -1)))
    .withColumn("rolling_std_7", F.stddev("sales_ff").over(w.rowsBetween(-7, -1)))
    .withColumn("rolling_std_28", F.stddev("sales_ff").over(w.rowsBetween(-28, -1)))
)

freq = (
    roll
    .withColumn("store_type_freq", F.count("*").over(Window.partitionBy("store_type")))
    .withColumn("product_family_freq", F.count("*").over(Window.partitionBy("product_family")))
)

features = (
    freq
    .withColumn("year", F.year("date"))
    .withColumn("month", F.month("date"))
    .withColumn("day", F.dayofmonth("date"))
    .withColumn("day_of_week", F.dayofweek("date"))
    .withColumn("week_of_year", F.weekofyear("date"))
    .withColumn("day_of_month", F.dayofmonth("date"))
    .withColumn("is_month_start", (F.dayofmonth("date") == 1).cast("int"))
    .withColumn("is_month_end", (F.last_day("date") == F.col("date")).cast("int"))
    .withColumn("is_weekend", (F.col("day_of_week").isin(1, 7)).cast("int"))
    .withColumn("is_payday", (F.col("day_of_month").isin(1, 15)).cast("int"))
    .withColumn("has_promotion", (F.col("onpromotion_ff") > 0).cast("int"))
    .withColumn("silver_timestamp", F.current_timestamp())
    .withColumn("oil_price", F.col("oil_price_ff"))
    .withColumn("is_national_holiday", F.col("is_national_holiday_ff").cast("int"))
    .withColumn("is_holiday", F.col("is_holiday_ff").cast("int"))
    .withColumn("is_event", F.col("is_event_ff").cast("int"))
    .withColumn("is_special_day", F.col("is_special_day_ff").cast("int"))
    .withColumn("daily_transactions", F.col("daily_transactions_ff").cast("int"))
)

features_clean = features.drop(
    "sales",
    "onpromotion",
    "oil_price_clean",
    "is_national_holiday_clean",
    "is_holiday_clean",
    "is_event_clean",
    "is_special_day_clean",
    "sales_ff",
    "onpromotion_ff",
    "oil_price_ff",
    "is_national_holiday_ff",
    "is_holiday_ff",
    "is_event_ff",
    "is_special_day_ff",
    "daily_transactions_ff"
)

In [0]:
train_features = (
    features_clean
    .join(
        sales.select("date", "store_nbr", "product_family", "id", "sales"),
        ["date", "store_nbr", "product_family"],
        "inner"
    )
)

spark.sql("DROP TABLE IF EXISTS store_sales_silver.silver_ml_features_train")

train_features.write.saveAsTable("store_sales_silver.silver_ml_features_train")

test_with_features = (
    features_clean
    .join(
        test.select("id", "date", "store_nbr", "product_family"),
        ["date", "store_nbr", "product_family"],
        "inner"
    )
)

spark.sql("DROP TABLE IF EXISTS store_sales_silver.silver_ml_features_test")

test_with_features.write.saveAsTable("store_sales_silver.silver_ml_features_test")


In [0]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor

import mlflow
import mlflow.xgboost

train_df = spark.table("store_sales_silver.silver_ml_features_train").toPandas()
test_df = spark.table("store_sales_silver.silver_ml_features_test").toPandas()

train_df["sales"] = pd.to_numeric(train_df["sales"], errors="coerce")
train_df = train_df.dropna(subset=["sales"])


categorical_cols = ["product_family", "store_type", "city", "state"]
category_maps = {}

for col in categorical_cols:
    train_df[col] = train_df[col].astype(str)
    test_df[col] = test_df[col].astype(str)

    unique_vals = sorted(train_df[col].unique())
    mapping = {v: i for i, v in enumerate(unique_vals)}
    category_maps[col] = mapping

    train_df[col] = train_df[col].map(mapping).astype("int32")

    test_df[col] = test_df[col].map(lambda v: mapping.get(v, -1)).astype("int32")

target_col = "sales"

non_feature_cols = [
    "id",
    "date",
    "silver_timestamp"
]

feature_cols = [
    c for c in train_df.columns
    if c not in non_feature_cols + [target_col]
]

X = train_df[feature_cols]
y = train_df[target_col]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

mlflow.xgboost.autolog(disable=True)

model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

y_pred = model.predict(X_val)
y_pred = np.maximum(y_pred, 0)

mse = mean_squared_error(y_val, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_val, y_pred)
r2 = r2_score(y_val, y_pred)

print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)

In [0]:
test_ids = test_df["id"].copy()
X_test = test_df[feature_cols]

test_predictions = model.predict(X_test)
test_predictions = np.maximum(test_predictions, 0)

inverse_maps = {
    col: {v: k for k, v in category_maps[col].items()}
    for col in categorical_cols
}

decoded_df = test_df.copy()

for col in categorical_cols:
    decoded_df[col] = decoded_df[col].map(inverse_maps[col])

decoded_df["sales_prediction"] = test_predictions

results_df = decoded_df[["id", "date", "store_nbr", "product_family", "sales_prediction"]]

In [0]:
predictions_spark = spark.createDataFrame(results_df)

spark.sql("CREATE SCHEMA IF NOT EXISTS store_sales_gold")
spark.sql("DROP TABLE IF EXISTS store_sales_gold.sales_predictions")

predictions_spark.write.saveAsTable("store_sales_gold.sales_predictions")

In [0]:
display(results_df)